In [1]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [2]:
import urllib.request
import json

def query_model(
    prompt,
    model="gemma3:27b",
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }


    # Convert the dictionary to a JSON formatted string and encode it to bytes
    payload = json.dumps(data).encode("utf-8")

    # Create a request object, setting the method to POST and adding necessary headers
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )
    request.add_header("Content-Type", "application/json")

    # Send the request and capture the response
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # Read and decode the response
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data


model = "llama3:latest"
result = query_model("Answer in one sentence: What do Llamas eat?", model=model)
print(result)

Llamas are herbivores and their diet typically consists of grasses, plants, leaves, and other vegetation, with a preference for fiber-rich foods like hay and grains.


In [3]:
# This cell is optional; it allows you to restart the notebook
# and only run section 7.7 without rerunning any of the previous code
import json

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)


def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [4]:
JUDGE_PROMPT = """
You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance.
You will be given an instruction, a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing the evaluation criteria.
Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
Please do not generate any other opening, closing, and explanations.

Here is the rubric you should use to build your answer:
1: The response fails to address the instructions, providing irrelevant, incorrect, or excessively verbose information that detracts from the user's request.
2: The response partially addresses the instructions but includes significant inaccuracies, irrelevant details, or excessive elaboration that detracts from the main task.
3: The response follows the instructions with some minor inaccuracies or omissions. It is generally relevant and clear, but may include some unnecessary details or could be more concise.
4: The response adheres to the instructions, offering clear, accurate, and relevant information in a concise manner, with only occasional, minor instances of excessive detail or slight lack of clarity.
5: The response fully adheres to the instructions, providing a clear, accurate, and relevant answer in a concise and efficient manner. It addresses all aspects of the request without unnecessary details or elaboration

Provide your feedback as follows:

Feedback:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the instruction, the reference answer, and the response.

Instruction: {instruction}
Reference Answer: {reference}
Answer: {answer}


Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """

In [5]:
for entry in test_data[:3]:
    prompt = JUDGE_PROMPT.format(
        instruction=format_input(entry),
        reference=entry['output'],
        answer=entry["model_response"]
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt, model=model))
    print("\n-------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a bullet train.

Score:
>> Feedback:::
The response successfully rewrites the sentence using a simile, comparing the car's speed to that of a bullet train. The comparison is relevant and clear, making it easy to understand the intended meaning.

Total rating: 4

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The type of cloud is a cirrus.

Score:
>> Feedback:::
The response fails to address the instructions by providing an incorrect answer that does not match the reference answer. The instruction specifically asks for the type of cloud typically associated with thunderstorms, but the response provides a different type of cloud altogether (cirrus).

Evaluation: The response is irrelevant and inaccurate, failing to provide any information related to the requested topic.

Total rating: 1

----------

In [6]:
from tqdm import tqdm
import re

def generate_model_scores(json_data, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = JUDGE_PROMPT.format(
            instruction=format_input(entry),
            reference=entry['output'],
            answer=entry["model_response"]
        )
        response = query_model(prompt, model)
        try:
            score_match = re.search(r'Total rating:\s*(\d+)', response)
            if score_match:
                score = int(score_match.group(1))
                scores.append(score)
            else:
                print(f"Could not find score in response: {response}")
                continue
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, model=model)
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:  58%|██████████████████████████████████████████████████████████████████▉                                                | 64/110 [01:42<01:25,  1.87s/it]

Could not find score in response: Feedback:::
The response fails to address the instructions by providing an incorrect and irrelevant answer. The question asks for the plural form of 'child', but the response suggests that it is 'young', which is not a grammatically correct or relevant plural form.

Evaluation: The response does not meet the criteria for any rating above 1, as it provides an incorrect and irrelevant answer that fails to address the main task.


Scoring entries: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 110/110 [03:00<00:00,  1.64s/it]

Number of scores: 109 of 110
Average score: 2.14

